# StealthVision — Detección de Armas (Módulo Futuro)
### knife · pistol — Fine-tuning con dataset propio

---

> **Este cuadernillo está listo para ejecutar cuando se quiera activar el módulo de armas.**  
> El módulo de tráfico (`01_TrafficRisk_Pipeline.ipynb`) es el principal para la presentación.

## Dataset disponible

| Split | Imágenes | Clases |
|---|---|---|
| Train | **4,409** | knife, pistol |
| Valid | **1,043** | knife, pistol |
| Test  | **385**   | knife, pistol |

Formatos: COCO JSON (`guns-knives-coco/`) y YOLO (`guns-knives-yolo/`)

## Estrategia

Entrenar **YOLOv8n** (nano, muy rápido) o **YOLOv8s** (small) en este dataset.  
Exportar a ONNX y correr en paralelo al RT-DETR de tráfico.

```
Frame
  ├── RT-DETR ONNX  → personas, vehículos, semáforos  (tráfico)
  └── YOLOv8 ONNX   → knife, pistol                  (armas)
           ↓
     CombinedRiskEngine
```

## Celda 1 — Exploración del Dataset

In [ ]:
import json, os, sys
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

# Rutas del dataset
COCO_ROOT = "guns-knives-coco/guns-knives-coco"
YOLO_ROOT = "guns-knives-yolo/guns-knives-yolo"

# Cargar anotaciones COCO
splits = {}
for split in ["train", "valid", "test"]:
    ann_path = os.path.join(COCO_ROOT, split, "_annotations.coco.json")
    with open(ann_path, encoding="utf-8") as f:
        splits[split] = json.load(f)

print("Dataset guns-knives")
print("=" * 40)
for split, data in splits.items():
    n_img = len(data["images"])
    n_ann = len(data["annotations"])
    print(f"{split:6s}: {n_img:5d} imágenes, {n_ann:5d} anotaciones")

print("\nCategorías:")
for cat in splits["train"]["categories"]:
    print(f"  id={cat['id']}  nombre={cat['name']}")

In [ ]:
# Distribucion de clases en train
cat_id_to_name = {c["id"]: c["name"] for c in splits["train"]["categories"]}
ann_by_class = Counter(a["category_id"] for a in splits["train"]["annotations"])

names  = [cat_id_to_name[k] for k in sorted(ann_by_class)]
counts = [ann_by_class[k]   for k in sorted(ann_by_class)]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(names, counts, color=["tomato", "steelblue", "orange"])
ax.set_title("Distribución de anotaciones (train)")
ax.set_ylabel("Cantidad")
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            str(int(bar.get_height())), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Mostrar 6 imágenes de muestra con sus bboxes
train_data = splits["train"]
id_to_ann  = {}
for ann in train_data["annotations"]:
    id_to_ann.setdefault(ann["image_id"], []).append(ann)

# Tomar imagenes que tengan anotaciones
imgs_with_ann = [img for img in train_data["images"] if img["id"] in id_to_ann]
samples = imgs_with_ann[:6]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

COLORS = {0: (255,0,0), 1: (0,200,0), 2: (0,0,255)}

for i, img_info in enumerate(samples):
    img_path = os.path.join(COCO_ROOT, "train", img_info["file_name"])
    img = cv2.imread(img_path)
    if img is None:
        axes[i].text(0.5, 0.5, "No encontrada", ha='center')
        axes[i].axis('off')
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    for ann in id_to_ann.get(img_info["id"], []):
        x, y, w, h = [int(v) for v in ann["bbox"]]  # COCO: x,y,w,h
        cid = ann["category_id"]
        color_rgb = tuple(reversed(COLORS.get(cid, (200,200,200))))
        import matplotlib.patches as mpatches
        rect = mpatches.Rectangle((x,y), w, h,
                                    linewidth=2, edgecolor=[c/255 for c in color_rgb],
                                    facecolor='none')
        axes[i].add_patch(rect)
        axes[i].text(x, y-5, cat_id_to_name[cid],
                     color=[c/255 for c in color_rgb], fontsize=9, fontweight='bold')

    axes[i].imshow(img_rgb)
    axes[i].axis('off')
    axes[i].set_title(img_info["file_name"][:30], fontsize=7)

plt.suptitle("Muestras del dataset guns-knives (COCO)", fontweight='bold')
plt.tight_layout()
plt.show()

## Celda 2 — Instalar Ultralytics (YOLOv8)
> Solo ejecutar si no está instalado

In [ ]:
# Verificar si ultralytics está instalado
try:
    import ultralytics
    print("Ultralytics ya instalado:", ultralytics.__version__)
except ImportError:
    print("Instalando ultralytics...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics"], check=True)
    import ultralytics
    print("Instalado:", ultralytics.__version__)

# Verificar data.yaml
yaml_path = os.path.join(YOLO_ROOT, "data.yaml")
print("\ndata.yaml:")
with open(yaml_path) as f:
    print(f.read())

## Celda 3 — Preparar data.yaml con rutas absolutas
YOLOv8 necesita rutas absolutas o relativas al directorio de trabajo

In [ ]:
import yaml

# Crear data.yaml con rutas absolutas correctas para este equipo
abs_root = os.path.abspath(YOLO_ROOT)

data_config = {
    "path":  abs_root,
    "train": os.path.join(abs_root, "train", "images"),
    "val":   os.path.join(abs_root, "valid", "images"),
    "test":  os.path.join(abs_root, "test",  "images"),
    "nc":    2,
    "names": ["knife", "pistol"],
}

# Guardar en la raiz del proyecto para facil acceso
LOCAL_YAML = "weapons_data.yaml"
with open(LOCAL_YAML, "w") as f:
    yaml.dump(data_config, f, default_flow_style=False, allow_unicode=True)

print(f"weapons_data.yaml creado:")
with open(LOCAL_YAML) as f:
    print(f.read())

# Verificar que las carpetas de imagenes existen
for split in ["train", "valid", "test"]:
    img_dir = os.path.join(abs_root, split, "images")
    lbl_dir = os.path.join(abs_root, split, "labels")
    imgs = len([f for f in os.listdir(img_dir) if f.endswith(('.jpg','.png'))]) if os.path.exists(img_dir) else 0
    lbls = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')]) if os.path.exists(lbl_dir) else 0
    print(f"{split:6s}: {imgs} imágenes, {lbls} labels")

## Celda 4 — Entrenamiento YOLOv8

Modelos disponibles:
- `yolov8n.pt` — nano (más rápido, ~10 min GPU)
- `yolov8s.pt` — small (~20 min GPU, mejor precisión)
- `yolov8m.pt` — medium (~45 min GPU)

Para la presentación, **yolov8s** es el equilibrio ideal.

In [ ]:
from ultralytics import YOLO

# Configuracion del entrenamiento
MODEL_BASE   = "yolov8s.pt"   # cambiar a yolov8n.pt para entrenamiento mas rapido
DATA_YAML    = "weapons_data.yaml"
EPOCHS       = 50             # 50 epochs es suficiente para este dataset
IMG_SIZE     = 640
BATCH_SIZE   = 16             # bajar a 8 si hay error de memoria GPU
PROJECT_NAME = "weapons_detector"
RUN_NAME     = "guns_knives_yolov8s"

print(f"Configuracion de entrenamiento:")
print(f"  Modelo base:  {MODEL_BASE}")
print(f"  Dataset:      {DATA_YAML}")
print(f"  Epochs:       {EPOCHS}")
print(f"  Batch size:   {BATCH_SIZE}")
print(f"  Image size:   {IMG_SIZE}")
print(f"\nResultados se guardarán en: runs/detect/{RUN_NAME}/")
print("\nEjecuta la celda siguiente para iniciar el entrenamiento.")
print("NOTA: puede tardar 15-45 minutos dependiendo de tu GPU.")

In [ ]:
# ============================================================
# EJECUTAR ESTA CELDA PARA INICIAR EL ENTRENAMIENTO
# ============================================================
from ultralytics import YOLO

model = YOLO(MODEL_BASE)

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=PROJECT_NAME,
    name=RUN_NAME,
    patience=15,          # early stopping si no mejora
    save=True,
    device=0,             # GPU 0, cambiar a 'cpu' si no hay GPU
    workers=4,
    verbose=True,
    augment=True,         # data augmentation automatico
    cos_lr=True,          # cosine learning rate scheduler
)

print("\nEntrenamiento completado.")
print(f"Mejor modelo guardado en: {results.save_dir}")

## Celda 5 — Evaluar el modelo entrenado

In [ ]:
import glob

# Buscar el mejor modelo entrenado
best_candidates = glob.glob(f"{PROJECT_NAME}/{RUN_NAME}/weights/best.pt")
if not best_candidates:
    best_candidates = glob.glob(f"{PROJECT_NAME}/*/weights/best.pt")

if best_candidates:
    BEST_MODEL = best_candidates[-1]
    print(f"Modelo encontrado: {BEST_MODEL}")

    model_eval = YOLO(BEST_MODEL)
    metrics = model_eval.val(data=DATA_YAML, split="test")

    print("\n=== Resultados en Test Set ===")
    print(f"mAP50:     {metrics.box.map50:.4f}")
    print(f"mAP50-95:  {metrics.box.map:.4f}")
    print(f"Precision: {metrics.box.mp:.4f}")
    print(f"Recall:    {metrics.box.mr:.4f}")
else:
    print("No se encontró modelo entrenado. Ejecuta la celda de entrenamiento primero.")

## Celda 6 — Exportar a ONNX

In [ ]:
if 'BEST_MODEL' not in dir() or not os.path.exists(BEST_MODEL):
    # Buscar automaticamente
    candidates = glob.glob("weapons_detector/*/weights/best.pt")
    BEST_MODEL = candidates[-1] if candidates else None

if BEST_MODEL:
    print(f"Exportando {BEST_MODEL} a ONNX...")
    model_export = YOLO(BEST_MODEL)
    export_path = model_export.export(
        format="onnx",
        imgsz=640,
        half=False,
        simplify=True,
        opset=16,
    )
    # Copiar a raiz del proyecto
    import shutil
    dest = "weapons-yolov8s.onnx"
    shutil.copy(export_path, dest)
    size_mb = os.path.getsize(dest) / 1e6
    print(f"\nONNX guardado: {dest} ({size_mb:.1f} MB)")
    print("Listo para integrar en el pipeline de StealthVision.")
else:
    print("No se encontró modelo. Ejecuta el entrenamiento primero.")

## Celda 7 — Probar el modelo ONNX exportado

In [ ]:
import onnxruntime as ort
import numpy as np
import cv2

WEAPONS_ONNX = "weapons-yolov8s.onnx"
WEAPONS_CLASSES = ["knife", "pistol"]
CONF_THRESH = 0.5

if not os.path.exists(WEAPONS_ONNX):
    print(f"No se encontró {WEAPONS_ONNX}. Ejecuta la celda de exportación primero.")
else:
    session = ort.InferenceSession(WEAPONS_ONNX,
                                    providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
    input_name = session.get_inputs()[0].name
    _, _, in_h, in_w = session.get_inputs()[0].shape
    print(f"Modelo: {WEAPONS_ONNX}")
    print(f"Input:  {in_w}x{in_h}")
    print(f"Backend: {session.get_providers()[0]}")

    def detect_weapons(image_path):
        """Detecta armas en una imagen usando el modelo ONNX exportado."""
        img = cv2.imread(image_path)
        if img is None:
            print(f"No se pudo leer: {image_path}"); return

        oh, ow = img.shape[:2]
        scale = min(in_w/ow, in_h/oh)
        nw, nh = int(ow*scale), int(oh*scale)
        canvas = np.full((in_h, in_w, 3), 114, np.uint8)
        canvas[:nh,:nw] = cv2.resize(img, (nw,nh))
        inp = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
        inp = np.expand_dims(inp.transpose(2,0,1), 0)

        # YOLOv8 ONNX output: (1, 6, N) — x1,y1,x2,y2,conf,class
        # (la forma exacta depende de la version; ajustar si es necesario)
        out = session.run(None, {input_name: inp})[0]

        print(f"\nImagen: {image_path} ({ow}x{oh})")
        print(f"Output shape: {out.shape}")

        # Parsear salida
        if out.ndim == 3:
            out = out[0]  # (6, N) o (N, 6)
        if out.shape[0] == 6:
            out = out.T   # → (N, 6)

        dets = []
        for row in out:
            x1,y1,x2,y2,conf,cls = row[:6]
            if float(conf) < CONF_THRESH: continue
            # Reescalar
            x1=max(0,int(x1/scale)); y1=max(0,int(y1/scale))
            x2=min(ow-1,int(x2/scale)); y2=min(oh-1,int(y2/scale))
            cname = WEAPONS_CLASSES[int(cls)] if int(cls)<len(WEAPONS_CLASSES) else str(int(cls))
            dets.append({"bbox":[x1,y1,x2,y2], "conf":round(float(conf),4), "class":cname})
            print(f"  {cname:8s} conf={conf:.3f}  bbox=[{x1},{y1},{x2},{y2}]")

        if not dets:
            print("  No se detectaron armas.")

        # Visualizar
        vis = img.copy()
        WCOLORS = {"knife": (0,200,0), "pistol": (0,0,255)}
        for d in dets:
            x1,y1,x2,y2 = d["bbox"]
            c = WCOLORS.get(d["class"], (200,200,200))
            cv2.rectangle(vis, (x1,y1), (x2,y2), c, 2)
            cv2.putText(vis, f"{d['class']} {d['conf']:.2f}",
                        (x1,y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, c, 2)

        plt.figure(figsize=(10,6))
        plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.title(f"Detección de Armas — {os.path.basename(image_path)}")
        plt.show()
        return dets

    # Probar con una imagen del dataset
    test_imgs = [f for f in os.listdir(os.path.join(COCO_ROOT, "test")) if f.endswith('.jpg')]
    if test_imgs:
        test_path = os.path.join(COCO_ROOT, "test", test_imgs[0])
        detect_weapons(test_path)

## Celda 8 — Integración futura: pipeline combinado

Cuando el módulo de armas esté entrenado, el pipeline completo será:

```python
# En main.py — versión futura
traffic_dets = traffic_detector.detect(image)    # RT-DETR: personas, vehículos
weapons_dets = weapons_detector.detect(image)    # YOLOv8: knife, pistol

all_dets = traffic_dets + weapons_dets
risk = combined_risk_engine.analyze(all_dets, image.shape)
```

Escenarios nuevos que se habilitarán:

| Escenario | Descripción educativa |
|---|---|
| Arma visible en zona pública | Explicar por qué es peligroso |
| Persona con arma cerca de otros | Máxima alerta |
| Arma sin persona visible | Zona de riesgo, no acercarse |


In [ ]:
# Esquema del CombinedRiskEngine (para implementar cuando el modelo de armas esté listo)

WEAPON_SCENARIOS = {
    "arma_y_persona": {
        "risk_level":  "CRITICAL",
        "titulo":      "¡PELIGRO EXTREMO! Arma cerca de personas",
        "explicacion": "Se detectó un arma muy cerca de una o más personas. Esta es una situación de máxima peligrosidad.",
        "leccion":     "Si ves un arma en un lugar público: NO te acerques. Busca a un adulto o llama a las autoridades.",
        "consejo":     "Aléjate inmediatamente. Llama al 911.",
    },
    "arma_sola": {
        "risk_level":  "CRITICAL",
        "titulo":      "¡PELIGRO! Arma detectada en la zona",
        "explicacion": "Se detectó un arma en el área. Las armas son objetos peligrosos que solo deben manejar autoridades capacitadas.",
        "leccion":     "Nunca toques un arma que encuentres. Avisa a un adulto o a la policía.",
        "consejo":     "No toques. No te acerques. Avisa a un adulto.",
    },
}

print("Escenarios de armas definidos:")
for k, v in WEAPON_SCENARIOS.items():
    print(f"  {k}: [{v['risk_level']}] {v['titulo']}")

print("\nEste modulo quedara pendiente hasta que el modelo YOLOv8 sea entrenado.")
print("Una vez entrenado, se integra en ~30 minutos al pipeline principal.")